# 04 – Data Preparation

This notebook covers:
- Loading `df_product_final` and `reviews_tfidf_df` from `s3://ads508-team1-sephora/Model/final_features/`
- Defining target variables (rating) for products and reviews
- Train / validation / test splitting (90 / 5 / 5)
- Saving all splits to `s3://ads508-team1-sephora/Model/final_splits/`

**Input:** `s3://ads508-team1-sephora/Model/final_features/`
**Output:** `s3://ads508-team1-sephora/Model/final_splits/`


## 1. Imports


In [1]:
import os
import boto3
import shutil
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

BUCKET             = "ads508-team1-sephora"
S3_FINAL_FEATURES  = "Model/final_features"
S3_FINAL_SPLITS    = "Model/final_splits"
LOCAL_TMP          = "./temp_splits"

os.makedirs(LOCAL_TMP, exist_ok=True)

## 2. Load Feature Matrices from S3

Both datasets were produced and saved by `03_data_transformation_engineering.ipynb`.

In [2]:
# Load the final feature matrices saved by NB03
df_product_final = pd.read_parquet(
    f"s3://{BUCKET}/{S3_FINAL_FEATURES}/df_product_final.parquet"
)
reviews_tfidf_df = pd.read_parquet(
    f"s3://{BUCKET}/{S3_FINAL_FEATURES}/reviews_tfidf_df.parquet"
)

print(f"df_product_final shape : {df_product_final.shape}")
print(f"reviews_tfidf_df shape : {reviews_tfidf_df.shape}")

df_product_final shape : (2369, 287)
reviews_tfidf_df shape : (10000, 3002)


## 3. Define Features & Targets

### 3.1 Product Targets

We predict two outcomes:
- **Rating regression**: `y_rating` = original product `rating` (loaded from curated, aligned to Makeup filter)
- **Loves regression**: `y_loves` = `log_loves` (re-engineered from curated for alignment)

In [3]:
# Load curated products to extract target columns
# (these were dropped from df_product_final in NB03)
products_curated = pd.read_parquet(
    f"s3://{BUCKET}/curated/products/products.parquet"
)
products_makeup = (
    products_curated[products_curated["primary_category"] == "Makeup"]
    .copy()
    .reset_index(drop=True)
)

y_rating_prod = products_makeup["rating"].values
y_loves_prod  = np.log1p(products_makeup["loves_count"].values)

X_products = df_product_final

print(f"X_products shape    : {X_products.shape}")
print(f"y_rating_prod shape : {y_rating_prod.shape}")
print(f"y_loves_prod shape  : {y_loves_prod.shape}")

X_products shape    : (2369, 287)
y_rating_prod shape : (2369,)
y_loves_prod shape  : (2369,)


### 3.2 Review Targets

In [4]:
X_reviews_feat = reviews_tfidf_df.drop(columns=["rating"])
y_rating_rev   = reviews_tfidf_df["rating"].values

print(f"X_reviews_feat shape : {X_reviews_feat.shape}")
print(f"y_rating_rev shape   : {y_rating_rev.shape}")

X_reviews_feat shape : (10000, 3001)
y_rating_rev shape   : (10000,)


## 4. Train / Validation / Test Split (90 / 5 / 5)

In [5]:
RANDOM_STATE   = 42
VAL_TEST_RATIO = 0.10   # 10% held out → split equally into val & test (5% each)

def split_dataset(X, y, random_state=RANDOM_STATE):
    """Return (X_train, X_val, X_test, y_train, y_val, y_test)."""
    X_train, X_tmp, y_train, y_tmp = train_test_split(
        X, y, test_size=VAL_TEST_RATIO, random_state=random_state
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_tmp, y_tmp, test_size=0.5, random_state=random_state
    )
    return X_train, X_val, X_test, y_train, y_val, y_test

# Products → rating target
X_train_r, X_val_r, X_test_r, y_train_r, y_val_r, y_test_r = split_dataset(
    X_products, y_rating_prod
)

# Reviews → rating target
X_train_rv, X_val_rv, X_test_rv, y_train_rv, y_val_rv, y_test_rv = split_dataset(
    X_reviews_feat, y_rating_rev
)

for name, arr in [
    ("X_train_r",  X_train_r),  ("X_val_r",  X_val_r),  ("X_test_r",  X_test_r),
    ("X_train_rv", X_train_rv), ("X_val_rv", X_val_rv), ("X_test_rv", X_test_rv),
]:
    print(f"{name:15s}: {arr.shape}")

X_train_r      : (2132, 287)
X_val_r        : (118, 287)
X_test_r       : (119, 287)
X_train_rv     : (9000, 3001)
X_val_rv       : (500, 3001)
X_test_rv      : (500, 3001)


## 5. Save Splits to S3

In [6]:
s3_client = boto3.client("s3")

splits = {
    # Product splits
    "product_X_train": pd.DataFrame(X_train_r),
    "product_X_val":   pd.DataFrame(X_val_r),
    "product_X_test":  pd.DataFrame(X_test_r),
    "product_y_train": pd.DataFrame(y_train_r, columns=["rating"]),
    "product_y_val":   pd.DataFrame(y_val_r,   columns=["rating"]),
    "product_y_test":  pd.DataFrame(y_test_r,  columns=["rating"]),
    # Review splits
    "review_X_train":  pd.DataFrame(X_train_rv),
    "review_X_val":    pd.DataFrame(X_val_rv),
    "review_X_test":   pd.DataFrame(X_test_rv),
    "review_y_train":  pd.DataFrame(y_train_rv, columns=["rating"]),
    "review_y_val":    pd.DataFrame(y_val_rv,   columns=["rating"]),
    "review_y_test":   pd.DataFrame(y_test_rv,  columns=["rating"]),
}

for name, df in splits.items():
    local_path = f"{LOCAL_TMP}/{name}.parquet"
    s3_key     = f"{S3_FINAL_SPLITS}/{name}.parquet"
    df.to_parquet(local_path, index=False, engine="pyarrow")
    s3_client.upload_file(local_path, BUCKET, s3_key)
    print(f"Uploaded  s3://{BUCKET}/{s3_key}  {df.shape}")

shutil.rmtree(LOCAL_TMP)
print("\nAll splits saved and temp directory cleaned up.")

Uploaded  s3://ads508-team1-sephora/Model/final_splits/product_X_train.parquet  (2132, 287)
Uploaded  s3://ads508-team1-sephora/Model/final_splits/product_X_val.parquet  (118, 287)
Uploaded  s3://ads508-team1-sephora/Model/final_splits/product_X_test.parquet  (119, 287)
Uploaded  s3://ads508-team1-sephora/Model/final_splits/product_y_train.parquet  (2132, 1)
Uploaded  s3://ads508-team1-sephora/Model/final_splits/product_y_val.parquet  (118, 1)
Uploaded  s3://ads508-team1-sephora/Model/final_splits/product_y_test.parquet  (119, 1)
Uploaded  s3://ads508-team1-sephora/Model/final_splits/review_X_train.parquet  (9000, 3001)
Uploaded  s3://ads508-team1-sephora/Model/final_splits/review_X_val.parquet  (500, 3001)
Uploaded  s3://ads508-team1-sephora/Model/final_splits/review_X_test.parquet  (500, 3001)
Uploaded  s3://ads508-team1-sephora/Model/final_splits/review_y_train.parquet  (9000, 1)
Uploaded  s3://ads508-team1-sephora/Model/final_splits/review_y_val.parquet  (500, 1)
Uploaded  s3://ads

## 6. Verification

In [7]:
# Reload one split from each dataset and confirm shape
checks = {
    "product_X_train": X_train_r.shape,
    "review_X_train":  X_train_rv.shape,
}
for name, expected_shape in checks.items():
    reloaded = pd.read_parquet(f"s3://{BUCKET}/{S3_FINAL_SPLITS}/{name}.parquet")
    status   = "✓" if reloaded.shape == expected_shape else "✗ MISMATCH"
    print(f"{name:25s}  expected={expected_shape}  reloaded={reloaded.shape}  {status}")

product_X_train            expected=(2132, 287)  reloaded=(2132, 287)  ✓
review_X_train             expected=(9000, 3001)  reloaded=(9000, 3001)  ✓


## Summary

| Step | Input | Output |
|------|-------|--------|
| Load features | `Model/final_features/df_product_final.parquet` | `X_products`, `y_rating_prod`, `y_loves_prod` |
| Load features | `Model/final_features/reviews_tfidf_df.parquet` | `X_reviews_feat`, `y_rating_rev` |
| Split (90/5/5) | Feature matrices | 12 train/val/test arrays |
| Save splits | Arrays | `Model/final_splits/*.parquet` |

➡ Ready for model training.